In [1]:
import os, sys
PROJECT_ROOT = "/home/yudai/Project/research/Vul_Detection"
os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)

print("cwd:", os.getcwd())


cwd: /home/yudai/Project/research/Vul_Detection


In [2]:
# ==== boot ====
import os, sys, torch
PROJECT_ROOT = "/home/yudai/Project/research/Vul_Detection"
if os.getcwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# (任意) 3090向け高速化
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ==== imports ====
import configs
from models.LMGNN import BertRGCN            # 必要に応じてクラス名はあなたの実装に合わせてください
from models.layers import Conv                    # ← ここから Conv を取る
ConvClass = Conv                      # ← あなたの Conv 実装

# ==== 1. モデルを構築 ====
# configs 側のオブジェクト（BertGGNN or BertModelCfg どちらか）
cfg = configs.BertGGNN() if hasattr(configs, "BertGGNN") else configs.BertModelCfg()

model_cfg = getattr(cfg, "model", {})
gated_graph_conv_args = dict(model_cfg.get("gated_graph_conv_args", {}))
conv_args             = dict(model_cfg.get("conv_args", {}))
emb_size              = model_cfg.get("emb_size", getattr(cfg, "emb_size", 768))

# ★ 必須：Conv を明示的に注入（モデルがこのキーを要求）
gated_graph_conv_args["Conv"] = ConvClass
conv_args["Conv"] = ConvClass

# （RGCN系で num_relations が必須ならここで指定）
# conv_args.setdefault("num_relations", 3)  # AST/CFG/PDG で 3 など。必要なら外して設定。

model = BertRGCN(gated_graph_conv_args, conv_args, emb_size, device).to(device)

# ==== 2. パラメータ一覧を表示 ====
print("\n===== model.parameters() の一覧 =====\n")
total_params = 0
trainable_params = 0

for name, param in model.named_parameters():
    print(f"Name: {name:<40} | Shape: {list(param.shape)} | requires_grad={param.requires_grad}")
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print("\n===== パラメータ集計 =====")
print(f"総パラメータ数        : {total_params:,}")
print(f"訓練可能パラメータ数  : {trainable_params:,}")
print(f"凍結（更新されない）パラメータ数: {total_params - trainable_params:,}")


device: cuda

===== model.parameters() の一覧 =====

Name: input_proj.weight                        | Shape: [200, 769] | requires_grad=True
Name: input_proj.bias                          | Shape: [200] | requires_grad=True
Name: rgcn_layers.0.weight                     | Shape: [3, 200, 200] | requires_grad=True
Name: rgcn_layers.0.root                       | Shape: [200, 200] | requires_grad=True
Name: rgcn_layers.0.bias                       | Shape: [200] | requires_grad=True
Name: rgcn_layers.1.weight                     | Shape: [3, 200, 200] | requires_grad=True
Name: rgcn_layers.1.root                       | Shape: [200, 200] | requires_grad=True
Name: rgcn_layers.1.bias                       | Shape: [200] | requires_grad=True
Name: rgcn_layers.2.weight                     | Shape: [3, 200, 200] | requires_grad=True
Name: rgcn_layers.2.root                       | Shape: [200, 200] | requires_grad=True
Name: rgcn_layers.2.bias                       | Shape: [200] | requires_gra